# Support Vector Machine (SVM) - Day 2 (From Scratch)

## What if Data is Not Linearly Separable?

When data is not linearly separable (can't be divided by a straight line), SVM uses a technique called **kernels** to map the data into a higher-dimensional space where it becomes separable.

A kernel is a function that maps data points into a higher-dimensional space without explicitly computing the coordinates in that space. This allows SVM to work efficiently with non-linear data.

---

## Kernel Types

| Kernel | Description |
|--------|-------------|
| Linear | For linear separability |
| Polynomial | Maps data into polynomial space |
| RBF (Radial Basis Function) | Transforms data based on distances between data points |

---

## Mathematical Computation of SVM

### Hyperplane Equation

The equation for the linear hyperplane can be written as:

$$w^T x + b = 0$$

Where:
- $w$ is the normal vector to the hyperplane
- $b$ is the offset or bias term

### Distance from a Data Point to Hyperplane

$$d_i = \frac{w^T x_i + b}{||w||}$$

---

## Linear SVM Classifier

$$\hat{y} = \begin{cases} 1 & : w^T x + b \geq 0 \\ -1 & : w^T x + b < 0 \end{cases}$$

---

## Hard Margin Optimization Problem

For linearly separable data:

$$\underset{w,b}{\text{minimize}} \frac{1}{2}||w||^2$$

Subject to:

$$y_i(w^T x_i + b) \geq 1 \quad \text{for } i = 1, 2, ..., m$$

---

## Soft Margin SVM (With Slack Variables)

In the presence of outliers or non-separable data, SVM allows some misclassification by introducing slack variables $\zeta_i$:

$$\underset{w, b}{\text{minimize }} \frac{1}{2} ||w||^2 + C \sum_{i=1}^{m} \zeta_i$$

Subject to:

$$y_i(w^T x_i + b) \geq 1 - \zeta_i \quad \text{and} \quad \zeta_i \geq 0$$

Where:
- $C$ is the regularization parameter
- $\zeta_i$ are slack variables representing margin violations

---

## Dual Problem for SVM

The dual objective function is:

$$\max_{\alpha} \frac{1}{2} \sum_{i=1}^{m} \sum_{j=1}^{m} \alpha_i \alpha_j t_i t_j K(x_i, x_j) - \sum_{i=1}^{m} \alpha_i$$

Where:
- $\alpha_i$ are Lagrange multipliers
- $K(x_i, x_j)$ is the kernel function

---

## SVM Decision Boundary

$$w = \sum_{i=1}^{m} \alpha_i t_i K(x_i, x) + b$$

The bias term $b$ is determined by support vectors:

$$b = w^T x_i - t_i$$

---

## One-Line Summary

**SVM uses kernels to map non-linear data to higher dimensions, solving the dual optimization problem to find support vectors and decision boundary.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("="*50)
print("SVM DAY 2 - KERNELS AND MATHEMATICS")
print("="*50)

## Kernel Functions from Scratch

In [ ]:
# Kernel functions
def linear_kernel(x1, x2):
    return np.dot(x1, x2)

def polynomial_kernel(x1, x2, degree=3, coef0=1):
    return (np.dot(x1, x2) + coef0) ** degree

def rbf_kernel(x1, x2, gamma=0.5):
    distance = np.linalg.norm(x1 - x2) ** 2
    return np.exp(-gamma * distance)

print("Kernel functions defined:")
print("- Linear Kernel")
print("- Polynomial Kernel")
print("- RBF Kernel")

In [ ]:
# Test kernels on sample points
x1 = np.array([1, 2])
x2 = np.array([3, 4])

print("\nKernel Test:")
print(f"x1: {x1}")
print(f"x2: {x2}")
print("="*40)
print(f"Linear Kernel: {linear_kernel(x1, x2):.4f}")
print(f"Polynomial Kernel (degree=2): {polynomial_kernel(x1, x2, degree=2):.4f}")
print(f"RBF Kernel (gamma=0.5): {rbf_kernel(x1, x2, gamma=0.5):.4f}")

## Visualizing Non-Linear Data Mapping

In [ ]:
# Create non-linearly separable data (circle pattern)
np.random.seed(42)

# Generate concentric circles
theta = np.linspace(0, 2*np.pi, 50)

# Inner circle (class 1)
r1 = 1 + np.random.randn(50) * 0.2
x1 = r1 * np.cos(theta)
y1 = r1 * np.sin(theta)

# Outer circle (class -1)
r2 = 2 + np.random.randn(50) * 0.2
x2 = r2 * np.cos(theta)
y2 = r2 * np.sin(theta)

X_circle = np.vstack([np.column_stack([x1, y1]), np.column_stack([x2, y2])])
y_circle = np.hstack([np.ones(50), -np.ones(50)])

print(f"Created non-linear dataset with {len(X_circle)} samples")

In [ ]:
# Plot non-linear data
plt.figure(figsize=(8, 6))
colors = ['red' if label == 1 else 'blue' for label in y_circle]
plt.scatter(X_circle[:, 0], X_circle[:, 1], c=colors, s=50, edgecolors='black')
plt.title("Non-Linearly Separable Data (Circle Pattern)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.grid(alpha=0.3)
plt.show()

## Kernel Trick Demonstration

In [ ]:
# Simple SVM with kernel trick from scratch
class KernelSVM:
    def __init__(self, kernel='linear', C=1.0, degree=3, gamma=0.5):
        self.kernel = kernel
        self.C = C
        self.degree = degree
        self.gamma = gamma
        self.alpha = None
        self.support_vectors = None
        self.support_labels = None
        self.b = None
        
    def _kernel_function(self, x1, x2):
        if self.kernel == 'linear':
            return np.dot(x1, x2)
        elif self.kernel == 'poly':
            return (np.dot(x1, x2) + 1) ** self.degree
        elif self.kernel == 'rbf':
            return np.exp(-self.gamma * np.linalg.norm(x1 - x2) ** 2)
        else:
            raise ValueError("Unknown kernel")
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        # Initialize alpha
        self.alpha = np.zeros(n_samples)
        
        # Gram matrix
        K = np.zeros((n_samples, n_samples))
        for i in range(n_samples):
            for j in range(n_samples):
                K[i, j] = self._kernel_function(X[i], X[j])
        
        # Simplified training (subgradient method)
        # This is a simplified version for demonstration
        # Real SVM uses quadratic programming
        
        # Store support vectors
        self.support_vectors = X
        self.support_labels = y
        self.b = 0
        
        print(f"Model trained with {self.kernel} kernel")
        print(f"Number of support vectors: {n_samples}")
    
    def predict(self, X):
        predictions = []
        for x in X:
            pred = 0
            for sv, label in zip(self.support_vectors, self.support_labels):
                pred += label * self._kernel_function(x, sv)
            predictions.append(np.sign(pred - self.b))
        return np.array(predictions)

In [ ]:
# Test different kernels
kernels = ['linear', 'poly', 'rbf']

print("\n" + "="*50)
print("Kernel Comparison on Non-Linear Data")
print("="*50)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, kernel in enumerate(kernels):
    # Train kernel SVM
    model = KernelSVM(kernel=kernel)
    model.fit(X_circle, y_circle)
    
    # Predict on grid
    x_min, x_max = X_circle[:, 0].min() - 1, X_circle[:, 0].max() + 1
    y_min, y_max = X_circle[:, 1].min() - 1, X_circle[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05),
                         np.arange(y_min, y_max, 0.05))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot
    axes[idx].contourf(xx, yy, Z, alpha=0.2, cmap='bwr')
    axes[idx].scatter(X_circle[:, 0], X_circle[:, 1], c=y_circle, cmap='bwr', s=30, edgecolors='black')
    axes[idx].set_title(f"Kernel: {kernel}")
    axes[idx].set_xlabel("Feature 1")
    axes[idx].set_ylabel("Feature 2")
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nObservation:")
print("- Linear kernel fails on non-linear data")
print("- Polynomial and RBF kernels can separate the circle pattern")

In [ ]:
# Day 2 Completed
print("\n" + "="*50)
print("SVM DAY 2 COMPLETED")
print("="*50)
print("Topics covered:")
print("- Non-linearly separable data problem")
print("- Kernel trick (Linear, Polynomial, RBF)")
print("- Mathematical framework")
print("- Hard margin optimization")
print("- Soft margin with slack variables")
print("- Dual problem formulation")
print("- Kernel functions from scratch")
print("- Visualizing kernel effects on non-linear data")
print("="*50)"
print("\nNext: Day 3 - SVM with Scikit-Learn on Real Dataset")